# 01 — CSV → Iceberg (Batch)

**Steps:**
1. Read `./data/orders.csv` with Spark (explicit schema)
2. Write to Iceberg table `iceberg_catalog.my_database.orders`
3. Read back from Iceberg to verify
4. Run SQL analytics on the table

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Spark-Developer").master("local[*]").getOrCreate()

spark

:: loading settings :: url = jar:file:/Users/saketkumar/Documents/github/helper/spark-dev/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/saketkumar/.ivy2.5.2/cache
The jars for the packages stored in: /Users/saketkumar/.ivy2.5.2/jars
org.apache.iceberg#iceberg-spark-runtime-4.0_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-868f2c83-062c-4f6e-8342-b4752494cf8a;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 in central
:: resolution report :: resolve 52ms :: artifacts dl 2ms
	:: modules in use:
	org.apache.iceberg#iceberg-spark-runtime-4.0_2.13;1.10.1 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	-------------------------

## Step 1 — Read CSV with explicit schema

In [2]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

schema = StructType([
    StructField("order_id",      IntegerType(), nullable=False),
    StructField("customer_name", StringType(),  nullable=False),
    StructField("product",       StringType(),  nullable=False),
    StructField("quantity",      IntegerType(), nullable=False),
    StructField("unit_price",    DoubleType(),  nullable=False),
    StructField("order_date",    StringType(),  nullable=False),
    StructField("status",        StringType(),  nullable=False),
])

orders_df = spark.read.csv("./data/orders.csv", header=True, schema=schema)
orders_df.printSchema()
orders_df.show(truncate=False)

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- order_date: string (nullable = true)
 |-- status: string (nullable = true)

+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pendin

## Step 2 — Write to Iceberg table

In [3]:
orders_df.writeTo("iceberg_catalog.my_database.orders") \
    .tableProperty("format-version", "2") \
    .createOrReplace()

print("Written to Iceberg table: iceberg_catalog.my_database.orders")

Written to Iceberg table: iceberg_catalog.my_database.orders


## Step 3 — Read back from Iceberg

In [4]:
iceberg_df = spark.table("iceberg_catalog.my_database.orders")
iceberg_df.show(truncate=False)
print(f"Total rows in Iceberg table: {iceberg_df.count()}")

+--------+-------------+------------+--------+----------+----------+---------+
|order_id|customer_name|product     |quantity|unit_price|order_date|status   |
+--------+-------------+------------+--------+----------+----------+---------+
|1       |Alice Johnson|Laptop      |1       |1299.99   |2024-01-15|completed|
|2       |Bob Smith    |Mouse       |2       |29.99     |2024-01-16|completed|
|3       |Carol White  |Keyboard    |1       |79.99     |2024-01-17|pending  |
|4       |David Lee    |Monitor     |1       |399.99    |2024-01-18|completed|
|5       |Eve Martinez |Headphones  |3       |149.99    |2024-01-19|shipped  |
|6       |Frank Brown  |Webcam      |2       |89.99     |2024-01-20|pending  |
|7       |Grace Kim    |USB Hub     |1       |49.99     |2024-01-21|completed|
|8       |Henry Wilson |Laptop Stand|2       |59.99     |2024-01-22|shipped  |
|9       |Iris Chen    |SSD Drive   |1       |119.99    |2024-01-23|completed|
|10      |James Davis  |Microphone  |1       |199.99

## Step 4 — SQL analytics

In [5]:
# Revenue by status
spark.sql("""
    SELECT
        status,
        COUNT(*)                              AS total_orders,
        SUM(quantity)                         AS total_units,
        ROUND(SUM(quantity * unit_price), 2)  AS total_revenue
    FROM iceberg_catalog.my_database.orders
    GROUP BY status
    ORDER BY total_revenue DESC
""").show()

+---------+------------+-----------+-------------+
|   status|total_orders|total_units|total_revenue|
+---------+------------+-----------+-------------+
|completed|           5|          6|      1929.94|
|  shipped|           2|          5|       569.95|
|  pending|           3|          4|       459.96|
+---------+------------+-----------+-------------+



In [6]:
# Iceberg snapshot history
spark.sql("SELECT snapshot_id, committed_at, operation FROM iceberg_catalog.my_database.orders.snapshots").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|4114315725319939862|2026-04-25 05:33:29.211|overwrite|
|59412497846581937  |2026-04-25 08:55:24.351|overwrite|
+-------------------+-----------------------+---------+

